In [9]:
import pandas as pd
import numpy as np
import gzip
import json
from datetime import datetime
from datetime import date
import plotly
import plotly.express as px 
from ast import literal_eval
import ast

### Write Xalt data to json file with key variables (libA, linkA, userT, userDT, envT[LOADEDMODULES, LD_LIBRARY_PATH])

In [ ]:


file_path = '/Users/tpapka/Summer2025/xalt/data/POLARIS.XALT-RUNDATA-20241215.gz'

target_keys = ['libA', 'linkA', 'userT', 'userDT', 'envT']

records = []

with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            try:
                obj = json.loads(line)
                extracted = {}

                for key in target_keys:
                    if key in obj:
                        if key == "envT":
                           env_data = obj.get("envT")
                           if isinstance(env_data, dict):
                              extracted["envT"] = {
                                    "LOADEDMODULES": env_data.get("LOADEDMODULES"),
                                    "LD_LIBRARY_PATH": env_data.get("LD_LIBRARY_PATH")
                              }
                        else:
                            extracted[key] = obj[key]

                if extracted:
                    records.append(extracted)

            except json.JSONDecodeError:
                continue  

output_file = "extracted_records.json"
with open(output_file, "w") as out_f:
    json.dump(records, out_f, indent=4)

print(f"Wrote {len(records)} JSON records to: {output_file}")


### Create TXT file for jobs ran on Xalt that data (To test for matching data)

In [ ]:

df = pd.read_json('extracted_records.json')
df['job_id'] = df['userT'].apply(lambda x: x.get('job_id') if isinstance(x, dict) else None)
unique_job_ids = df['job_id'].nunique()
print(f"Number of unique job IDs: {unique_job_ids}")
job = df['job_id'].unique()

job = np.delete(job,0)
with open('xalt_job.txt', 'w') as f:
    for item in job:
        f.write(str(item)+ '\n') 

Number of unique job IDs: 469


### Load PySnooper and Polaris data also exploding libA from Xalt

In [ ]:

#need polaris and pysnooper 2024 data
pysnooper2024 = pd.read_csv('/Users/tpapka/Summer2025/PyModuleSnooper/pysnooper2024.csv')
polaris_data = pd.read_csv('/Users/tpapka/Summer2025/PyModuleSnooper/polaris_job_location_20250609.csv')

test = pysnooper2024[pysnooper2024['date'] == '2024-12-15']
test['JOB_NAME'] = test['JOB_NAME'].map(lambda x: x.rstrip('.polaris'))


xaltTest = pd.read_json('extracted_records.json')
xaltTest['JOB_NAME'] = xaltTest['userT'].apply(lambda x: x.get('job_id'))

pysnooperXaltComparisson = pd.merge(test,xaltTest, on = 'JOB_NAME')
# pysnooperXaltComparisson['libA'] = pysnooperXaltComparisson['libA'].explode().reset_index()

pysnooperXaltComparissonLibExplode = pysnooperXaltComparisson.explode('libA')
for i in pysnooperXaltComparissonLibExplode['libA']:
   i.pop()


def listToString(x):
   if isinstance(x, list) and len(x) == 1:
      return x[0]
   return x
pysnooperXaltComparissonLibExplode['libA'] = pysnooperXaltComparissonLibExplode['libA'].apply(listToString)
pysnooperXaltComparissonLibExplode['libA'] = pysnooperXaltComparissonLibExplode['libA'].apply(lambda x: x.split('/')[-1].split('.so')[0])



/var/folders/33/hsn1szm57t10kpxgfznwvncw0000gn/T/ipykernel_75726/666037400.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  polaris_data = pd.read_csv('/Users/tpapka/Summer2025/PyModuleSnooper/polaris_job_location_20250609.csv')
/var/folders/33/hsn1szm57t10kpxgfznwvncw0000gn/T/ipykernel_75726/666037400.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['JOB_NAME'] = test['JOB_NAME'].map(lambda x: x.rstrip('.polaris'))


### Counting jobs between Xalt and Pysnooper and graphing total jobs from [Polaris, Pysnooper, Xalt]

In [ ]:


PYdec20241215 = pysnooper2024[pysnooper2024['date'] == '2024-12-15']
PYdec20241215['JOB_NAME'].unique()
polaris_data['date'] = pd.to_datetime(polaris_data['START_TIMESTAMP']).dt.date

polaris_data
polarisdec20241215 = polaris_data[polaris_data['date'] == date(2024, 12, 15)]
PYdec20241215_test = PYdec20241215['JOB_NAME'].unique()
polaris_test = polarisdec20241215['JOB_NAME'].unique()

# for i in PYdec20241215_test:
#    if i in polaris_test:
#       count += 1
polaris_jobs = [s.split('.')[0] for s in polaris_test]
pysnooper_jobs = [s.split('.')[0]for s in PYdec20241215_test]

with open("xalt_job.txt", "r") as file:  
    xalt = file.readlines()
count = 0  
for i in xalt:
    if i.strip() in pysnooper_jobs:
        count += 1
print(count)
pysnooper_jobs

job_comparison = { 'Dataset' : ['Polaris','PySnooper', 'Xalt'],
    'Job_Count': [451,109,468]}
job_comparison = pd.DataFrame(job_comparison)

fig = px.bar(job_comparison, x = 'Dataset', y = 'Job_Count', title = 'Unique Jobs Ran 12-15-24', labels = {
    'Job_Count': 'Unique Job Count'
})
fig.show()
fig.write_html('/Users/tpapka/Summer2025/xalt/Viz/UniqueJobExampleBarChart.html')



109


AttributeError: module 'plotly' has no attribute 'bar'

### Graphing Top 15 Libraries from Xalt and Top 15 Modules from PySnooper

In [ ]:

pysnooperXaltComparissonModuleExplode = pysnooperXaltComparisson.copy()
pysnooperXaltComparissonModuleExplode['modules'] = pysnooperXaltComparissonModuleExplode['modules'].apply(literal_eval)
pysnooperXaltComparissonModuleExplode = pysnooperXaltComparissonModuleExplode.explode('modules').reset_index()

lib_count = pysnooperXaltComparissonLibExplode.groupby('libA')['libA'].count().rename('Lib Count').reset_index()
lib_count_top15 = lib_count.sort_values(by = 'Lib Count', ascending=False).head(15)

barXalt = px.bar(lib_count_top15,x = 'libA',y = 'Lib Count', title = 'Library Count From Xalt 12-15-24 (Top 15)', labels = {
   'libA':'Library',
   'Lib Count':'Count'
})
barXalt.show()

module_count = pysnooperXaltComparissonModuleExplode.groupby('modules')['modules'].count().rename('Module Count').reset_index()
module_count_top15 = module_count.sort_values(by = 'Module Count', ascending = False).head(15)


barPySnooper = px.bar(module_count_top15, x = 'modules', y = 'Module Count', title = 'Module Count From PySnooper 12-15-24 (Top 15)', labels = {
   'modules':'Modules',
})
barPySnooper.show()

charts = [('barXalt', barXalt),('barPySnooper', barPySnooper)]
for name, fig in charts:
   fig.write_html(f'/Users/tpapka/Summer2025/xalt/Viz/{name}Dec152024Top15.html')